# 2. Merge Single-Cell Profiles

## Purpose
This notebook reads the per-compartment DuckDB produced by notebook 1 for a single
well-FOV and merges the Nuclei, Cell, and Cytoplasm tables into a single-cell (SC)
parquet profile. Organoid and Nucleocentric profiles are passed through and saved as
separate parquets.

This is **step 2 of Stage 4 (image-based profiling)**. It runs once per well-FOV and
is typically submitted as a child job via the SLURM scheduler.

## Inputs
- `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/{well_fov}.duckdb`
  - Five compartment tables: `Organoid`, `Nuclei`, `Cell`, `Cytoplasm`, `Nucleocentric`
  - Produced by notebook 1 (`1.merge_feature_parquets.ipynb`)

## Outputs
Three parquet files written to `data/{patient}/image_based_profiles/0.converted_profiles/{well_fov}/`:

| File | Content | Rows |
|---|---|---|
| `sc_profiles_{well_fov}.parquet` | Merged Nuclei + Cell + Cytoplasm features | One row per object present in all three compartments |
| `organoid_profiles_{well_fov}.parquet` | Organoid features passed through | One row per segmented organoid |
| `nucleocentric_profiles_{well_fov}.parquet` | Nucleocentric features passed through | One row per nucleus-centered volume |

## Notes
- Only objects present in **all three** of Nuclei, Cell, and Cytoplasm are retained in the SC profile.
  Objects segmented in only some compartments are dropped.
- Object IDs are reassigned to a sequential `1..N` range at the end of this notebook.
  The original segmentation mask IDs are not preserved.

In [1]:
import os
import pathlib
import sys

import duckdb
import numpy as np
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    # patient = "NF0055_T1"
    # well_fov = "D5-1"
    patient = "NF0014_T1"
    well_fov = "C4-1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
input_sqlite_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/{well_fov}.duckdb"
).resolve(strict=True)
destination_sc_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve()
destination_organoid_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve()
destination_nucleocentric_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
destination_sc_parquet_file.parent.mkdir(parents=True, exist_ok=True)

# for empty tables:
merged_example_df_path = pathlib.Path(
    f"{root_dir}/4.processing_image_based_profiles/data/DB_structures/single_cell_profile_structure.parquet"
).resolve()
nucleocentric_example_df_path = pathlib.Path(
    f"{root_dir}/4.processing_image_based_profiles/data/DB_structures/nucleocentric_profile_structure.parquet"
).resolve()
organoid_example_df_path = pathlib.Path(
    f"{root_dir}/4.processing_image_based_profiles/data/DB_structures/organoid_profile_structure.parquet"
).resolve()
organoid_example_df_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
try:
    # Load all five compartment tables from the DuckDB produced by notebook 1.
    with duckdb.connect(input_sqlite_file) as con:
        tables = con.execute("SHOW TABLES").fetchdf()
        print(tables)
        nuclei_table = con.sql("SELECT * FROM Nuclei").df()
        cells_table = con.sql("SELECT * FROM Cell").df()
        cytoplasm_table = con.sql("SELECT * FROM Cytoplasm").df()
        organoid_table = con.sql("SELECT * FROM Organoid").df()
        nucleocentric_table = con.sql("SELECT * FROM Nucleocentric").df()
except Exception as e:
    if "Catalog" in str(e):
        print(
            f"Error: DuckDB file {input_sqlite_file} does not contain expected tables."
        )
        print("Writing empty DataFrames to output parquet files.")
        merged_df = pd.read_parquet(merged_example_df_path)
        nucleocentric_df = pd.read_parquet(nucleocentric_example_df_path)
        organoid_df = pd.read_parquet(organoid_example_df_path)

        # add the well_fov and objects
        merged_df = pd.concat(
            [
                pd.DataFrame(
                    {
                        "well_fov": [well_fov],
                        "object_id": [-1],
                        **{
                            col: [np.nan]
                            for col in merged_df.columns
                            if col not in ["well_fov", "object_id"]
                        },
                    }
                ),
            ]
        )
        nucleocentric_df = pd.concat(
            [
                pd.DataFrame(
                    {
                        "well_fov": [well_fov],
                        "object_id": [-1],
                        **{
                            col: [np.nan]
                            for col in nucleocentric_df.columns
                            if col not in ["well_fov", "object_id"]
                        },
                    }
                ),
            ]
        )
        organoid_df = pd.concat(
            [
                pd.DataFrame(
                    {
                        "well_fov": [well_fov],
                        "object_id": [-1],
                        **{
                            col: [np.nan]
                            for col in organoid_df.columns
                            if col not in ["well_fov", "object_id"]
                        },
                    }
                ),
            ]
        )

        merged_df.to_parquet(destination_sc_parquet_file, index=False)
        nucleocentric_df.to_parquet(destination_nucleocentric_parquet_file, index=False)
        organoid_df.to_parquet(destination_organoid_parquet_file, index=False)
    # exit the script after writing the empty DataFrames
    sys.exit(0)

            name
0           Cell
1      Cytoplasm
2         Nuclei
3  Nucleocentric
4       Organoid


In [5]:
# Retain only objects that were successfully segmented in all three compartments.
# A nucleus without a matched cell/cytoplasm (or vice versa) is not a valid
# single-cell profile and is dropped here.
nuclei_id_set = set(nuclei_table["object_id"].to_list())
cells_id_set = set(cells_table["object_id"].to_list())
cytoplasm_id_set = set(cytoplasm_table["object_id"].to_list())

# find the intersection of the three sets
intersection_set = nuclei_id_set.intersection(cells_id_set, cytoplasm_id_set)

# keep only the rows in the three tables that are in the intersection set
nuclei_table = nuclei_table[nuclei_table["object_id"].isin(intersection_set)]
cells_table = cells_table[cells_table["object_id"].isin(intersection_set)]
cytoplasm_table = cytoplasm_table[cytoplasm_table["object_id"].isin(intersection_set)]

In [6]:
# Merge the three compartment tables into a single-cell dataframe.
# Because object_ids were already filtered to the intersection in the cell above,
# this LEFT JOIN is effectively an INNER JOIN — no NaN-filled rows will result.
with duckdb.connect() as con:
    con.register("nuclei", nuclei_table)
    con.register("cells", cells_table)
    con.register("cytoplasm", cytoplasm_table)
    # Merge them with SQL
    merged_df = con.execute("""
        SELECT *
        FROM nuclei
        LEFT JOIN cells USING (object_id)
        LEFT JOIN cytoplasm USING (object_id)
    """).df()

In [7]:
# save the organoid data as parquet
print(f"Final organoid data shape: {organoid_table.shape}")
organoid_table.to_parquet(destination_organoid_parquet_file, index=False)
organoid_table.head()

Final organoid data shape: (1, 3961)


,object_id,image_set,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,Organoid_NoChannel_AreaSizeShape_MinY,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C4-1,27421038.0,711.460877,936.634612,19.864556,50391450.0,267,1082,109,...,35.276193,36.687325,35.246483,36.669919,35.24153,35.241639,35.290421,36.683411,35.280867,35.286644


In [8]:
print(f"Final merged single cell dataframe shape: {merged_df.shape}")
# save the sc data as parquet
merged_df.to_parquet(destination_sc_parquet_file, index=False)
merged_df.head()

Final merged single cell dataframe shape: (55, 11883)


,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_DNA_Texture_Variance-3-03-256,Cytoplasm_DNA_Texture_Variance-3-04-256,Cytoplasm_DNA_Texture_Variance-3-05-256,Cytoplasm_DNA_Texture_Variance-3-06-256,Cytoplasm_DNA_Texture_Variance-3-07-256,Cytoplasm_DNA_Texture_Variance-3-08-256,Cytoplasm_DNA_Texture_Variance-3-09-256,Cytoplasm_DNA_Texture_Variance-3-10-256,Cytoplasm_DNA_Texture_Variance-3-11-256,Cytoplasm_DNA_Texture_Variance-3-12-256
0,257,C4-1,7480.0,881.128610,443.802139,1.488770,10200.0,857,907,418,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,514,C4-1,37965.0,570.619781,889.462874,4.402107,53088.0,533,612,848,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,771,C4-1,45736.0,630.840716,980.077226,4.681083,80442.0,576,685,939,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1028,C4-1,32804.0,513.994940,1233.819229,3.297586,55692.0,471,562,1178,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1285,C4-1,78836.0,805.882478,657.815478,9.888135,139956.0,752,859,604,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
print(f"Final nucleocentric dataframe shape: {nucleocentric_table.shape}")
# save the nucleocentric data as parquet
nucleocentric_table.to_parquet(destination_nucleocentric_parquet_file, index=False)
nucleocentric_table.head()

Final nucleocentric dataframe shape: (55, 3074)


,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature90,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99
0,257,C4-1,-0.076046,-0.195801,0.099514,-0.077355,-0.087526,0.241190,0.049520,-0.119339,...,0.200481,4.058367,-5.435637,6.323337,6.208817,2.223289,0.309957,4.362774,1.527360,4.466059
1,514,C4-1,-0.393783,-0.264406,0.186608,-0.027248,-0.126305,0.140763,0.055341,-0.074079,...,1.480416,2.492021,-7.538130,3.304373,4.828958,3.199276,3.434937,3.212957,2.097705,0.973808
2,771,C4-1,-0.087094,-0.134559,0.149383,0.022436,-0.080138,0.239864,0.091013,-0.101817,...,0.671647,3.752697,-9.147327,-0.890167,5.157977,0.981803,1.429000,4.665015,0.730875,-1.203775
3,1028,C4-1,-0.203926,-0.255455,0.170370,-0.029354,-0.167938,0.139980,0.130247,-0.072525,...,0.857458,1.644536,-11.373195,-2.486522,7.482898,0.459792,-0.277286,5.905641,-2.663505,-0.668626
4,1285,C4-1,-0.201375,-0.211282,0.138028,-0.002726,-0.114167,0.264720,0.034140,-0.104014,...,1.908465,5.684974,-6.584369,5.397185,2.600838,1.398547,-0.746936,4.803525,3.831946,2.683017


In [10]:
# if patient=NF0014_T1 and well_fov=C4-1, then output zero's out dfs to DB_structures
if patient == "NF0014_T1" and well_fov == "C4-1":
    merged_df = pd.DataFrame(columns=merged_df.columns)
    nucleocentric_df = pd.DataFrame(columns=nucleocentric_table.columns)
    organoid_df = pd.DataFrame(columns=organoid_table.columns)
    merged_df.to_parquet(merged_example_df_path, index=False)
    nucleocentric_df.to_parquet(nucleocentric_example_df_path, index=False)
    organoid_df.to_parquet(organoid_example_df_path, index=False)